# **Import**


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from data_loader import Dataset
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

# **Build From Scratch**


In [2]:
class Node():

    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, benefit=None, value=None):

        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.benefit = benefit

        self.value = value

In [3]:
class XGBTree():

    def __init__(self, max_depth, min_child_weight, gamma, reg_lambda):

        self.root = None

        self.max_depth = max_depth
        self.min_child_weight = min_child_weight

        self.gamma = gamma
        self.reg_lambda = reg_lambda

    def _compute_all_g_i(self, y, y_pred):

        return y_pred - y

    def _compute_all_h_i(self, y, y_pred):

        if len(np.unique(y)) == 2:
            return y_pred * (1 - y_pred)
        else:
            return np.ones_like(y)

    def _build_tree(self, dataset_idxs, curr_depth=0):

        if curr_depth < self.max_depth:

            best_decision = self._get_best_decision(dataset_idxs)

            if best_decision and best_decision["benefit"] > 0 and best_decision["child_weight"] > self.min_child_weight:

                child_l = self._build_tree(best_decision["dataset_left_idxs"], curr_depth + 1)
                child_r = self._build_tree(best_decision["dataset_right_idxs"], curr_depth + 1)

                return Node(best_decision["feature_idx"], best_decision["threshold"], child_l, child_r, best_decision["benefit"])

        leaf_value = self._compute_leaf_value(dataset_idxs)

        return Node(value = leaf_value)

    def _get_best_decision(self, dataset_idxs, k=10):

        best_decision = {}

        max_benefit = -float("inf")

        num_features = self.X[dataset_idxs].shape[1]

        for feature_idx in range(num_features):

            feature_values = self.X[dataset_idxs, feature_idx]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds[::k]:

                dataset_left_idxs, dataset_right_idxs = self._split(dataset_idxs, feature_idx, threshold)

                if len(dataset_left_idxs) > 0 and len(dataset_right_idxs) > 0:

                    curr_benefit = self._compute_benefit(dataset_idxs, dataset_left_idxs, dataset_right_idxs)

                    curr_child_weight = self._compute_weight(dataset_left_idxs, dataset_right_idxs)

                    if curr_benefit > max_benefit:
                        best_decision["feature_idx"] = feature_idx
                        best_decision["threshold"] = threshold
                        best_decision["dataset_left_idxs"] = dataset_left_idxs
                        best_decision["dataset_right_idxs"] = dataset_right_idxs
                        best_decision["benefit"] = curr_benefit
                        max_benefit = curr_benefit

                        best_decision["child_weight"] = curr_child_weight

        return best_decision

    def _split(self, dataset_idxs, feature_idx, threshold):

        feature_values = self.X[dataset_idxs, feature_idx]

        is_left = feature_values <= threshold

        dataset_left_idxs = dataset_idxs[is_left]
        dataset_right_idxs = dataset_idxs[~is_left]

        return dataset_left_idxs, dataset_right_idxs

    def _compute_weight(self, l_idxs, r_idxs):

        H_l = self.all_h_i[l_idxs].sum()
        H_r = self.all_h_i[r_idxs].sum()

        return min(H_l, H_r)

    def _compute_benefit(self, parent_idxs, l_idxs, r_idxs):

        G_parent = self.all_g_i[parent_idxs].sum()
        G_l = self.all_g_i[l_idxs].sum()
        G_r = self.all_g_i[r_idxs].sum()

        H_parent = self.all_h_i[parent_idxs].sum()
        H_l = self.all_h_i[l_idxs].sum()
        H_r = self.all_h_i[r_idxs].sum()

        J_parent = self._compute_leaf_objective(G_parent, H_parent)
        J_l = self._compute_leaf_objective(G_l, H_l)
        J_r = self._compute_leaf_objective(G_r, H_r)

        return J_parent - (J_l + J_r) - self.gamma

    def _compute_leaf_objective(self, G_j, H_j):

        return -0.5 * ((G_j ** 2) /(H_j + self.reg_lambda))

    def _compute_leaf_value(self, idxs):

        G_j = self.all_g_i[idxs].sum()
        H_j = self.all_h_i[idxs].sum()

        return -G_j / (H_j + self.reg_lambda)

    def fit(self, X, y, y_pred):

        self.X = X
        self.y = y

        self.all_g_i = self._compute_all_g_i(y, y_pred)
        self.all_h_i = self._compute_all_h_i(y, y_pred)

        dataset_idxs = np.arange(len(y))

        self.root = self._build_tree(dataset_idxs)

    def predict(self, X):

        predictions = [self._predict(x, self.root) for x in X]

        return np.array(predictions)

    def _predict(self, x, tree):

        if tree.value is not None:
            return tree.value

        feature_val = x[tree.feature_idx]

        if feature_val <= tree.threshold:
            return self._predict(x, tree.left)

        else:
            return self._predict(x, tree.right)


In [4]:
class CustomXGBoost():

    def __init__(self,
                 n_estimators=100,
                 subsample=0.8,
                 colsample_bytree=0.8,
                 learning_rate=0.5,
                 reg_lambda=1.5,
                 gamma=0.05,
                 max_depth=3,
                 min_child_weight=5):

        self.wls = []
        self.feature_idxs = []

        self.n_estimators = n_estimators

        self.subsample = subsample
        self.colsample_bytree = colsample_bytree

        self.learning_rate = learning_rate
        self.reg_lambda = reg_lambda
        self.gamma = gamma

        self.max_depth = max_depth
        self.min_child_weight = min_child_weight

    def _compute_y_pred(self, F_h, prob=True):

        if self.problem_type == "binary classification":
            y_pred = 1 / (1 + np.exp(- F_h))

            if not prob:
                y_pred = np.array([1 if i >= 0.5 else 0 for i in y_pred])
        else:
            y_pred = F_h

        return y_pred

    def _compute_F0(self, y):

        if self.problem_type == "binary classification":
            return np.log(sum(y == 1) / sum(y == 0))
        else:
            return np.mean(y)

    def _generate_sample(self, X, y, y_pred):

        n, m = X.shape

        m_new = int(self.colsample_bytree * m)
        col_idxs = np.random.choice(m, m_new)

        n_new = int(self.subsample * n)
        row_idxs = np.random.choice(n, n_new)

        return X[row_idxs][:, col_idxs], y[row_idxs], y_pred[row_idxs], col_idxs

    def fit(self, X, y):

        if len(set(y)) == 2:
            self.problem_type = "binary classification"
        else:
            self.problem_type = "regression"

        self.F_0 = self._compute_F0(y)

        F_h = np.full_like(y, self.F_0, dtype="float")

        for _ in range(self.n_estimators):

            y_pred = self._compute_y_pred(F_h)

            X_sample, y_sample, y_pred_sample, feature_idxs = self._generate_sample(X, y, y_pred)

            wl = XGBTree(self.max_depth,self.min_child_weight,
                         self.gamma, self.reg_lambda)

            wl.fit(X_sample, y_sample, y_pred_sample)

            F_h += self.learning_rate * wl.predict(X[:, feature_idxs])

            self.wls.append(wl)
            self.feature_idxs.append(feature_idxs)

    def predict(self, X):

        F_k = self.F_0 + self.learning_rate * np.sum([wl.predict(X[:, idxs]) for wl, idxs in zip(self.wls, self.feature_idxs)], axis = 0)

        y_pred = self._compute_y_pred(F_k, prob = False)

        return y_pred


# **Load and Split**


In [5]:
dataset_b = Dataset("binary classification")
X_train_b, X_test_b, y_train_b, y_test_b = dataset_b.load_split_data()

dataset_r = Dataset("regression")
X_train_r, X_test_r, y_train_r, y_test_r = dataset_r.load_split_data()

# **Train, Test and Compare**



In [6]:
params = {
    "n_estimators": 100,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "learning_rate": 0.5,
    "reg_lambda": 1.5,
    "gamma": 0.05,

    "max_depth": 3,
    "min_child_weight": 5
}

In [7]:
custom_model_b = CustomXGBoost(**params)
custom_model_b.fit(X_train_b, y_train_b)
y_pred_custom_b = custom_model_b.predict(X_test_b)
print(f"Custom Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_custom_b):.3f}")

custom_model_r = CustomXGBoost(**params)
custom_model_r.fit(X_train_r, y_train_r)
y_pred_custom_r = custom_model_r.predict(X_test_r)
print(f"Custom Regression MSE: {mean_squared_error(y_test_r, y_pred_custom_r):.3f}")

Custom Binary Classification Accuracy: 0.942
Custom Regression MSE: 0.288


In [8]:
xgboost_model_b = XGBClassifier(**params)
xgboost_model_b.fit(X_train_b, y_train_b)
y_pred_xgboost_b = xgboost_model_b.predict(X_test_b)
print(f"XGBoost Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_xgboost_b):.3f}")

xgboost_model_r = XGBRegressor(**params)
xgboost_model_r.fit(X_train_r, y_train_r)
y_pred_xgboost_r = xgboost_model_r.predict(X_test_r)
print(f"XGBoost Regression MSE: {mean_squared_error(y_test_r, y_pred_xgboost_r):.3f}")

XGBoost Binary Classification Accuracy: 0.978
XGBoost Regression MSE: 0.262
